# Phase 4 — Create a Reproducible Tiny Subset

> **Rule:** Never pick images manually. Always use a fixed seed so the exact same 100 images are selected every single time.

**Goal:** Select exactly **50 Normal (Class 0)** and **50 Pneumonia (Class 1)** images from the training split using `SEED = 42`, and save a `subset_manifest.csv` that maps every selected image to its original index.

**Deliverable:** `ml/data/processed/subset_manifest.csv`

**Definition of done:** Running this notebook twice produces the identical manifest file.

In [1]:
# ── Step 0: Imports ───────────────────────────────────────────────────────────
import os
import hashlib
import numpy as np
import pandas as pd
import medmnist
from medmnist import INFO

# ── Config ────────────────────────────────────────────────────────────────────
SEED               = 42
SAMPLES_PER_CLASS  = 50
DATA_CACHE_DIR     = '../data/'
PROCESSED_DIR      = '../data/processed/'
MANIFEST_PATH      = os.path.join(PROCESSED_DIR, 'subset_manifest.csv')

os.makedirs(PROCESSED_DIR, exist_ok=True)
print(f'SEED={SEED} | samples_per_class={SAMPLES_PER_CLASS} | total={SAMPLES_PER_CLASS * 2}')

SEED=42 | samples_per_class=50 | total=100


In [2]:
# ── Step 1: Load Full Training Split Into Memory ───────────────────────────────
info = INFO['pneumoniamnist']
DataClass = getattr(medmnist, info['python_class'])

dataset = DataClass(split='train', download=True, root=DATA_CACHE_DIR)
all_images = dataset.imgs          # shape: (N, 28, 28)
all_labels = dataset.labels.squeeze()  # shape: (N,)

print(f'Full training split: {len(all_images)} images')
print(f'Label distribution : {dict(zip(*np.unique(all_labels, return_counts=True)))}')

Full training split: 4708 images
Label distribution : {np.uint8(0): np.int64(1214), np.uint8(1): np.int64(3494)}


In [3]:
# ── Step 2: Reproducible Sampling With Fixed Seed ─────────────────────────────
np.random.seed(SEED)

rows = []
for class_id in [0, 1]:
    class_indices  = np.where(all_labels == class_id)[0]
    chosen_indices = np.random.choice(class_indices, SAMPLES_PER_CLASS, replace=False)
    chosen_indices.sort()  # deterministic ordering within class

    for rank, original_idx in enumerate(chosen_indices):
        sample_id = f'class{class_id}_seed{SEED}_{rank:03d}'
        rows.append({
            'sample_id'      : sample_id,
            'original_index' : int(original_idx),
            'label'          : int(class_id),
            'split'          : 'train_subset',
        })

manifest = pd.DataFrame(rows)
print(f'Subset created: {len(manifest)} rows')
manifest.head(6)

Subset created: 100 rows


,sample_id,original_index,label,split
0,class0_seed42_000,211,0,train_subset
1,class0_seed42_001,215,0,train_subset
2,class0_seed42_002,319,0,train_subset
3,class0_seed42_003,349,0,train_subset
4,class0_seed42_004,453,0,train_subset
5,class0_seed42_005,492,0,train_subset


In [4]:
# ── Step 3: Save Manifest to CSV ──────────────────────────────────────────────
manifest.to_csv(MANIFEST_PATH, index=False)
print(f'✅ Manifest saved to {MANIFEST_PATH}')

# Compute a hash to prove reproducibility
with open(MANIFEST_PATH, 'rb') as f:
    file_hash = hashlib.md5(f.read()).hexdigest()

print(f'MD5 hash: {file_hash}')
print('Run this notebook again — the hash must be identical to prove reproducibility.')

✅ Manifest saved to ../data/processed/subset_manifest.csv
MD5 hash: d163ec32097f7eb2155d73d967757626
Run this notebook again — the hash must be identical to prove reproducibility.


In [5]:
# ── Step 4: Full Validation Report ────────────────────────────────────────────
loaded = pd.read_csv(MANIFEST_PATH)

total       = len(loaded)
class0_count = (loaded['label'] == 0).sum()
class1_count = (loaded['label'] == 1).sum()
duplicates   = loaded['original_index'].duplicated().sum()
missing_ids  = loaded['sample_id'].isnull().sum()

# Assert everything is correct
assert total == 100,            f'Expected 100 rows, got {total}'
assert class0_count == 50,      f'Expected 50 Class 0, got {class0_count}'
assert class1_count == 50,      f'Expected 50 Class 1, got {class1_count}'
assert duplicates == 0,         f'Found {duplicates} duplicate indices'
assert missing_ids == 0,        f'Found {missing_ids} missing sample IDs'

print('═══════════════════════════════════')
print('     SUBSET VALIDATION REPORT      ')
print('═══════════════════════════════════')
print(f'  Total samples : {total}')
print(f'  Class 0 (Normal)    : {class0_count}')
print(f'  Class 1 (Pneumonia) : {class1_count}')
print(f'  Duplicates    : {duplicates}')
print(f'  Missing IDs   : {missing_ids}')
print(f'  Seed used     : {SEED}')
print('═══════════════════════════════════')
print('✅ All assertions passed. Subset is valid and reproducible.')

═══════════════════════════════════
     SUBSET VALIDATION REPORT      
═══════════════════════════════════
  Total samples : 100
  Class 0 (Normal)    : 50
  Class 1 (Pneumonia) : 50
  Duplicates    : 0
  Missing IDs   : 0
  Seed used     : 42
═══════════════════════════════════
✅ All assertions passed. Subset is valid and reproducible.


In [6]:
# ── Step 5: Preview Manifest ──────────────────────────────────────────────────
print('First 5 rows:')
print(manifest[manifest['label'] == 0].head(3).to_string(index=False))
print()
print(manifest[manifest['label'] == 1].head(3).to_string(index=False))
print()
print(f'→ Full manifest at: {os.path.abspath(MANIFEST_PATH)}')

First 5 rows:
        sample_id  original_index  label        split
class0_seed42_000             211      0 train_subset
class0_seed42_001             215      0 train_subset
class0_seed42_002             319      0 train_subset

        sample_id  original_index  label        split
class1_seed42_000              30      1 train_subset
class1_seed42_001              42      1 train_subset
class1_seed42_002             291      1 train_subset

→ Full manifest at: /Volumes/Fullstack/Github/MedVision/ml/data/processed/subset_manifest.csv
